In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
import gensim.downloader as api
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch
import torch.nn as nn

In [ ]:
df = pd.read_csv('Musical_instruments_reviews.csv')
df = df[['reviewText', 'overall']].dropna()
df['reviewText'] = df['reviewText'].astype(str).str.lower()
# rating >= 4 is positive, else negative
df['label'] = (df['overall'] >= 4).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df['reviewText'], df['label'], test_size=0.2, random_state=0)

In [9]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

classif_tfidf = LogisticRegression(max_iter=1000, random_state=0)
classif_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = classif_tfidf.predict(X_test_tfidf)

precision_tfidf = precision_score(y_test, y_pred_tfidf)
recall_tfidf    = recall_score(y_test, y_pred_tfidf)
f1_tfidf        = f1_score(y_test, y_pred_tfidf)
print(f"TF-IDF: Precision: {precision_tfidf:.5f}, Recall: {recall_tfidf:.5f}, F1: {f1_tfidf:.5f}")

TF-IDF: Precision: 0.88517, Recall: 0.99722, F1: 0.93786


In [11]:
sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4, epochs=10)
w2v_vocab = w2v_model.wv.key_to_index

# compute average word vector for a sentence
def avg_word2vec(sentence):
    words = sentence.split()
    vecs = [w2v_model.wv[w] for w in words if w in w2v_vocab]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

# average embeddings for train and test
X_train_w2v = np.array([avg_word2vec(text) for text in X_train])
X_test_w2v = np.array([avg_word2vec(text) for text in X_test])

classif_w2v = LogisticRegression(max_iter=1000, random_state=0)
classif_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = classif_w2v.predict(X_test_w2v)
precision_w2v = precision_score(y_test, y_pred_w2v)
recall_w2v = recall_score(y_test, y_pred_w2v)
f1_w2v = f1_score(y_test, y_pred_w2v)
print(f"Word2Vec: Precision: {precision_w2v:.5f}, Recall: {recall_w2v:.5f}, F1: {f1_w2v:.5f}")

Word2Vec: Precision: 0.88536, Recall: 0.99056, F1: 0.93501


In [12]:
glove = api.load("glove-twitter-100")

def avg_glove(sentence):
    words = sentence.split()
    vecs = [glove[w] for w in words if w in glove.key_to_index]
    return np.mean(vecs, axis=0) if vecs else np.zeros(glove.vector_size)

X_train_glove = np.array([avg_glove(text) for text in X_train])
X_test_glove = np.array([avg_glove(text) for text in X_test])

classif_glove = LogisticRegression(max_iter=1000, random_state=0)
classif_glove.fit(X_train_glove, y_train)

y_pred_glove = classif_glove.predict(X_test_glove)
precision_glove = precision_score(y_test, y_pred_glove)
recall_glove = recall_score(y_test, y_pred_glove)
f1_glove = f1_score(y_test, y_pred_glove)
print(f"GloVe: Precision: {precision_glove:.5f}, Recall: {recall_glove:.5f}, F1: {f1_glove:.5f}")

[==================================================] 100.0% 387.1/387.1MB downloaded
GloVe: Precision: 0.88261, Recall: 0.99778, F1: 0.93667


In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

train_encodings = tokenize(list(X_train))
test_encodings  = tokenize(list(X_test))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [14]:
class dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = dataset(train_encodings, y_train.tolist())
test_dataset  = dataset(test_encodings,  y_test.tolist())


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DistilBert(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        pooled_output = hidden_state[:, 0]
        dropped = self.dropout(pooled_output)
        logits = self.classifier(dropped)
        return logits

model = DistilBert().to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [16]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        loss = loss_func(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Loss: {avg_loss:.5f}")


Epoch 1 - Loss: 0.29816
Epoch 2 - Loss: 0.19950
Epoch 3 - Loss: 0.11488


In [18]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels)

precision_bert = precision_score(all_labels, all_preds, average='binary')
recall_bert = recall_score(all_labels, all_preds, average='binary')
f1_bert = f1_score(all_labels, all_preds, average='binary')

print(f"BERT classifier: Precision: {precision_bert:.5f}, Recall: {recall_bert:.5f}, F1: {f1_bert:.5f}")


BERT classifier: Precision: 0.91219, Recall: 0.98057, F1: 0.94514
